# Shock Propagation in Pressureless SPH

In this tutorial, you will set up and run a **shock propagation test** using the pressure-less SPH model in Struphy, demonstrating particle-based shock capturing and comparing numerical results to analytical Burger's equation solutions.

This notebook focuses on the `PressureLessSPH` model applied to 1D inviscid shock problems. We will explore **two regimes**:

1. **Large-amplitude shock**: A Riemann shock with sharp density/velocity jump (standard test case from literature)
2. **Small-amplitude waves**: Linear sinusoidal perturbations where density evolution is minimal

The workflow demonstrates core Struphy concepts: environment setup, particle initialization, time integration, and diagnostics—all in interactive Python so you can adjust parameters and rerun sections quickly.

## Mathematical Background: Inviscid Burger's Equation

The pressureless continuity and momentum equations form a hyperbolic system:

$$
\begin{aligned}
  \partial_t \rho + \partial_x(\rho u) &= 0 \quad \text{(Continuity)} \\
  \partial_t(\rho u) + \partial_x(\rho u^2) &= 0 \quad \text{(Momentum without pressure)}
\end{aligned}
$$

For small-amplitude perturbations around a mean state, these equations support acoustic waves. For large-amplitude shocks, the system develops weak solutions with discontinuities.

### Riemann Problem (Large Amplitude)

We solve the Riemann problem with initial conditions:

$$
(\rho, u)(x, 0) = \begin{cases}
  (\rho_L, u_L) & \text{if } x < 0.5 \\
  (\rho_R, u_R) & \text{if } x > 0.5
\end{cases}
$$

The **Rankine-Hugoniot shock speed** is: $s = \frac{u_L + u_R}{2}$ (for this system).

### Small-Amplitude Waves (Linear Regime)

For perturbations of the form $u(x, 0) = \epsilon \sin(\pi x)$ with small $\epsilon \ll 1$, the density evolution is governed by:

$$
\partial_t \rho \approx -\rho_0 \partial_x u = -\rho_0 \epsilon \pi \cos(\pi x)
$$

Density changes remain $O(\epsilon)$ small, and the velocity evolves along characteristics:

$$
u(x, t) = \epsilon \sin(\pi(x - \epsilon \pi t)) + O(\epsilon^2)
$$

## Step 0: Import Struphy Components

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import cunumpy as xp

# Struphy imports
from struphy import (
    BaseUnits,
    EnvironmentOptions,
    Time,
    domains,
    equils,
    grids,
    DerhamOptions,
    BoundaryParameters,
    LoadingParameters,
    WeightsParameters,
    SortingParameters,
    SavingParameters,
    BinningPlot,
    Simulation,
)
from struphy.models import PressureLessSPH
from struphy.initial.base import Perturbation

## Step 1: Create Environment, Time Integrator, and 1D Domain

Set up a 1D periodic domain (stretched into 3D as required by Struphy) with Strang splitting for time integration.

In [ ]:
# ====== CONFIGURABLE PARAMETERS ======
# Time stepping
dt = 1.0e-3      # time step
Tend = 0.1       # end time

# Domain (1D periodic in logical coords eta1 ∈ [0, 1])
l1, r1 = 0.0, 1.0  # eta1 range
l2, r2 = 0.0, 1.0  # eta2 range (minimal extent)
l3, r3 = 0.0, 1.0  # eta3 range (minimal extent)
# =====================================

# Environment options
env = EnvironmentOptions(sim_folder="sim_shock_large", profiling_activated=False)

# Time stepping with Strang splitting
time_opts = Time(dt=dt, Tend=Tend, split_algo="Strang")

# Geometry: 1D periodic, extended to 3D cuboid
domain = domains.Cuboid(l1=l1, r1=r1, l2=l2, r2=r2, l3=l3, r3=r3)

print(f"Domain: eta1 ∈ [{l1}, {r1}], eta2 ∈ [{l2}, {r2}], eta3 ∈ [{l3}, {r3}]")
print(f"Time stepping: dt={dt}, Tend={Tend}")

## Step 2: Define Riemann Step for Shock Initial Condition

We implement a smooth approximation to the Riemann jump using a tanh transition. This avoids sharp discontinuities in the initial particle distribution while maintaining the shock structure.

In [ ]:
class RiemannStep(Perturbation):
    """Smooth approximation of a 1D Riemann jump in logical coordinate eta1.
    
    Args:
        left: value on the left side  (x < eta0)
        right: value on the right side (x > eta0)
        eta0: location of the jump (default 0.5 = middle of domain)
        width: width of tanh transition (smaller = sharper transition)
        given_in_basis: basis for given variable ("0" for density, "v" for velocity)
        comp: component index for vector quantities
    """

    def __init__(self, left: float, right: float, eta0: float = 0.5, 
                 width: float = 0.01, given_in_basis: str="0", comp: int = 0):
        self.left = left
        self.right = right
        self.eta0 = eta0
        self.width = width
        self.given_in_basis = given_in_basis
        self.comp = comp

    def __call__(self, e1, e2, e3):
        """Evaluate the smooth step at logical coordinates (e1, e2, e3)."""
        avg = 0.5 * (self.left + self.right)
        half_jump = 0.5 * (self.left - self.right)
        return avg - half_jump * xp.tanh((e1 - self.eta0) / self.width)


# Parameters for large-amplitude Riemann shock (Sod-like test case)
# Left state: high density, high velocity
rho_L, u_L = 1.0, 2.0
# Right state: low density, zero velocity
rho_R, u_R = 0.125, 0.0

print("\nLarge-amplitude Riemann shock test:")
print(f"  Left state:  ρ={rho_L}, u={u_L}")
print(f"  Right state: ρ={rho_R}, u={u_R}")
print(f"  Shock speed (Rankine-Hugoniot): s = (u_L + u_R)/2 = {(u_L + u_R)/2}")

## Step 3: Configure Grid and de Rham Discretization

Set up the tensor-product grid and de Rham options for the 1D problem. Since this is truly 1D physics in a 3D logical domain, we use minimal resolution in non-1D directions.

In [ ]:
# Grid: fine resolution in eta1, minimal in others
grid = grids.TensorProductGrid(num_elements=(64, 1, 1))

# de Rham options: periodic boundaries in all directions
derham_opts = DerhamOptions(degree=(3, 1, 1))

print("Grid elements: (64, 1, 1)")
print("Boundary conditions: periodic in all directions")

## Step 4: Instantiate the PressureLessSPH Model

Create a lightweight model instance for the first simulation (large-amplitude shock case).

In [ ]:
# Model instance (no external field for pure Burger's equation)
model = PressureLessSPH()

# Simulation object
sim = Simulation(
    model,
    env=env,
    time_opts=time_opts,
    domain=domain,
    equil=None,  # No background equilibrium needed
    grid=grid,
    derham_opts=derham_opts,
)

print("PressureLessSPH model instantiated (no external field).")

## Step 5: Configure Particle Markers with SPH Diagnostics

Set up particle loading, weights, boundaries, and binning diagnostics. For the shock test, we use:**

- **Np = 256** particles (can be tuned)
- **Periodic boundaries** in all directions (1D domain)
- **256 bins** for binning diagnostics to track density and velocity evolution

In [ ]:
# ====== CONFIGURABLE PARTICLE PARAMETERS ======
Np_particles = 10000  # Number of particles
n_bins = 64        # Number of bins for diagnostics
# ==============================================

loading_params = LoadingParameters(Np=Np_particles)
weights_params = WeightsParameters()

# Periodic boundaries in all directions (1D periodic domain)
boundary_params = BoundaryParameters(
    bc=("periodic", "periodic", "periodic"),
    bc_sph=("periodic", "periodic", "periodic")
)

# Sorting: 256 boxes in eta1 direction, minimal in others
sorting_params = SortingParameters(
    boxes_per_dim=(n_bins, 1, 1),
    dims_mask=(True, False, False)  # Only sort in eta1
)

# Binning diagnostics for density profile
bin_plot = BinningPlot(
    slice="e1",
    n_bins=(n_bins,),
    ranges=(0.0, 1.0),
    output_quantity="density"
)

saving_params = SavingParameters(binning_plots=(bin_plot,))

model.cold_fluid.set_markers(
    loading_params=loading_params,
    weights_params=weights_params,
    boundary_params=boundary_params,
    sorting_params=sorting_params,
    saving_params=saving_params,
)

print(f"Particle setup: Np={Np_particles}, {n_bins} bins for diagnostics")

## Step 6: Set Propagator Options (No External Field)

Configure the time integrators for particle position and velocity updates. **No external potential is applied** (φ = 0), so we have pure pressureless Burger's equation.

In [ ]:
from struphy import ButcherTableau

# Forward Euler time integration for both position and velocity
butcher = ButcherTableau(algo="forward_euler")
model.propagators.push_eta.options = model.propagators.push_eta.Options(butcher=butcher)

# No external field (phi=None means zero forcing)
model.propagators.push_v.options = model.propagators.push_v.Options(phi=None)

print("Propagators configured: forward_euler, no external field")

# Case 1: Large-Amplitude Riemann Shock

In this section, we set up and run the large-amplitude shock case, demonstrating shock capturing and propagation.

## Step 7a: Apply Riemann Shock Initial Conditions

In [ ]:
# Constant background (zero velocity, unit density)
background = equils.ConstantVelocity(ux=0.0, uy=0.0, uz=0.0, n=1.0, p0=0.0)
model.cold_fluid.var.add_background(background)

# Perturbations: density and velocity jumps (both smooth tanh transitions)
del_n = RiemannStep(
    left=rho_L - 1.0,      # density perturbation on left
    right=rho_R - 1.0,     # density perturbation on right
    eta0=0.5,
    width=0.01
)

del_u1 = RiemannStep(
    left=u_L,          # velocity on left
    right=u_R,         # velocity on right
    eta0=0.5,
    width=0.01,
    given_in_basis="v"  # given in velocity basis
)

model.cold_fluid.var.add_perturbation(del_n=del_n, del_u1=del_u1)

print("Riemann shock initial conditions applied:")
print("  Background: ρ=1.0, u=(0, 0, 0)")
print("  Perturbation: smooth tanh transition with width=0.01")

## Step 8a: Run Large-Amplitude Shock Simulation

In [ ]:
print("\n" + "="*60)
print("Running LARGE-AMPLITUDE RIEMANN SHOCK simulation...")
print("="*60)
sim.run()
print("Simulation completed.")

## Post-Process and Load Results for Large-Amplitude Case

In [ ]:
print("Post-processing...")
sim.pproc()
print("Loading plotting data...")
sim.load_plotting_data()
print("Data loaded.")

## Visualize Large-Amplitude Shock Evolution

Plot density profiles at multiple time snapshots to observe shock propagation and steepening.

In [ ]:
# Extract binning data
binning_data = sim.f.cold_fluid.e1_density.f_binned
t_grid = sim.t_grid
eta1_bins = np.linspace(0, 1, n_bins + 1)[:-1]  # bin centers

# Plot density evolution at selected time snapshots
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

# Select 6 time snapshots
Nt = t_grid.size
snapshot_indices = np.linspace(0, Nt - 1, 6, dtype=int)

for idx, (ax, t_idx) in enumerate(zip(axes, snapshot_indices)):
    print(binning_data.shape)
    rho_profile = binning_data[t_idx, :]
    ax.plot(eta1_bins, rho_profile, 'b-', linewidth=2)
    ax.axvline(0.5, color='gray', linestyle='--', alpha=0.5, label='Initial shock location')
    ax.set_xlabel(r'$\eta_1$ (logical coordinate)')
    ax.set_ylabel(r'$\rho$ (density)')
    ax.set_title(f'Large-amplitude shock at t = {t_grid[t_idx]:.4f}')
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 1.2])
    if idx == 0:
        ax.legend()

plt.tight_layout()
plt.suptitle('Density Evolution: Large-Amplitude Riemann Shock', fontsize=14, y=1.00)
plt.show()

print(f"\nScheme captured shock evolution from t=0 to t={t_grid[-1]}")
print(f"Expected shock propagation speed: s ≈ {(u_L + u_R)/2}")

# Case 2: Small-Amplitude Linear Waves

Now we demonstrate the small-amplitude regime where density changes are minimal and linear wave theory applies.

## Set Up Second Simulation with Small-Amplitude IC

Create a new model and simulation instance for the small-amplitude case, using a separate output folder.

In [ ]:
# Create new model instance for second simulation
model2 = PressureLessSPH()

# Environment for case 2 (separate folder)
env2 = EnvironmentOptions(sim_folder="sim_shock_small", profiling_activated=False)

# Simulation object (reuse domain, grid, time Options)
sim2 = Simulation(
    model2,
    env=env2,
    time_opts=time_opts,  # same time stepping
    domain=domain,         # same domain
    equil=None,
    grid=grid,
    derham_opts=derham_opts,
)

print("Second simulation instance created.")

## Configure Markers and Propagators for Case 2

In [ ]:
# Same marker configuration as case 1
loading_params2 = LoadingParameters(Np=Np_particles)
weights_params2 = WeightsParameters()
boundary_params2 = BoundaryParameters(
    bc=("periodic", "periodic", "periodic"),
    bc_sph=("periodic", "periodic", "periodic")
)
sorting_params2 = SortingParameters(
    boxes_per_dim=(n_bins, 1, 1),
    dims_mask=(True, False, False)
)

bin_plot2 = BinningPlot(
    slice="e1",
    n_bins=(n_bins,),
    ranges=(0.0, 1.0),
    output_quantity="density"
)
saving_params2 = SavingParameters(binning_plots=(bin_plot2,))

model2.cold_fluid.set_markers(
    loading_params=loading_params2,
    weights_params=weights_params2,
    boundary_params=boundary_params2,
    sorting_params=sorting_params2,
    saving_params=saving_params2,
)

# Same propagator options
butcher2 = ButcherTableau(algo="forward_euler")
model2.propagators.push_eta.options = model2.propagators.push_eta.Options(butcher=butcher2)
model2.propagators.push_v.options = model2.propagators.push_v.Options(phi=None)

print("Markers and propagators configured for case 2.")

## Step 7b: Apply Small-Amplitude Linear Wave IC

Initialize with a sinusoidal velocity perturbation: $u(x) = \epsilon \sin(\pi x)$ where $\epsilon$ is small.

In [ ]:
# ====== CONFIGURABLE AMPLITUDE ======
epsilon_amplitude = 0.1  # Small-amplitude parameter (can be tuned)
# ====================================

# Background: unit density, zero velocity
background2 = equils.ConstantVelocity(ux=0.0, uy=0.0, uz=0.0, n=1.0, p0=0.0)
model2.cold_fluid.var.add_background(background2)

# Linear wave perturbation: sinusoidal velocity in u1 component
class SinusoidalPerturbation(Perturbation):
    """Sinusoidal velocity perturbation: u(x) = epsilon * sin(pi * x)"""
    
    def __init__(self, amplitude: float, given_in_basis: str="v", comp: int = 0):
        self.amplitude = amplitude
        self.given_in_basis = given_in_basis
        self.comp = comp
    
    def __call__(self, e1, e2, e3):
        return self.amplitude * xp.sin(np.pi * e1)

# No density perturbation (background density = 1.0)
# Only velocity perturbation
del_u1_linear = SinusoidalPerturbation(amplitude=epsilon_amplitude, given_in_basis="v")
model2.cold_fluid.var.add_perturbation(del_u1=del_u1_linear)

print("Small-amplitude linear wave IC:")
print("  Background: ρ=1.0, u=(0, 0, 0)")
print(f"  Perturbation: u₁(η₁) = {epsilon_amplitude} × sin(π η₁)")
print("  Linear wave approximation valid for small ε")

## Step 8b: Run Small-Amplitude Simulation

In [ ]:
print("\n" + "="*60)
print("Running SMALL-AMPLITUDE LINEAR WAVE simulation...")
print("="*60)
sim2.run()
print("Simulation completed.")

## Post-Process Results for Small-Amplitude Case

In [ ]:
print("Post-processing case 2...")
sim2.pproc()
print("Loading plotting data...")
sim2.load_plotting_data()
print("Data loaded.")

## Visualize Small-Amplitude Wave Evolution

Plot density profiles showing minimal change in the linear regime.

In [ ]:
# Extract binning data for case 2
binning_data2 = sim2.f.cold_fluid.e1_density.f_binned
t_grid2 = sim2.t_grid
eta1_bins2 = np.linspace(0, 1, n_bins + 1)[:-1]

# Plot density evolution
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

Nt2 = t_grid2.size
snapshot_indices2 = np.linspace(0, Nt2 - 1, 6, dtype=int)

for idx, (ax, t_idx) in enumerate(zip(axes, snapshot_indices2)):
    rho_profile2 = binning_data2[t_idx, :]
    ax.plot(eta1_bins2, rho_profile2, 'g-', linewidth=2)
    ax.axhline(1.0, color='gray', linestyle='--', alpha=0.5, label='Background density')
    ax.set_xlabel(r'$\eta_1$ (logical coordinate)')
    ax.set_ylabel(r'$\rho$ (density)')
    ax.set_title(f'Small-amplitude wave at t = {t_grid2[t_idx]:.4f}')
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 1.5])  # Zoomed scale to show small variations
    if idx == 0:
        ax.legend()

plt.tight_layout()
plt.suptitle('Density Evolution: Small-Amplitude Linear Wave', fontsize=14, y=1.00)
plt.show()

# Compute density variation
rho_max = np.max(binning_data2)
rho_min = np.min(binning_data2)
rho_variation = (rho_max - rho_min) / 1.0  # relative to background

print("\nSmall-amplitude wave:")
print(f"  Density range: [{rho_min:.6f}, {rho_max:.6f}]")
print(f"  Relative density variation: {rho_variation*100:.3f}%")
print(f"  Expected O(ε) behavior: ε ≈ {epsilon_amplitude} → density variation ~ {epsilon_amplitude*100:.1f}%")

# Analytical Solutions and Comparison

Now we compute analytical solutions and compare them to the numerical results.

## Analytical Solution: Rankine-Hugoniot Shock

For the large-amplitude shock, the weak solution is a moving discontinuity. The shock speed is given by the Rankine-Hugoniot condition:

$$
s = \frac{u_L + u_R}{2}
$$

The weak solution is:

$$
(\rho, u)(x, t) = \begin{cases}
  (\rho_L, u_L) & \text{if } x < s \cdot t + x_0 \\
  (\rho_R, u_R) & \text{if } x > s \cdot t + x_0
\end{cases}
$$

where $x_0 = 0.5$ is the initial shock location.

In [ ]:
# Compute shock position at final time
shock_speed = (u_L + u_R) / 2
t_final = t_grid[-1]
x0_shock = 0.5
shock_pos_final = x0_shock + shock_speed * t_final

# Account for periodicity
shock_pos_final = shock_pos_final % 1.0

print("\nAnalytical Shock Solution:")
print(f"  Shock speed: s = (u_L + u_R)/2 = {shock_speed}")
print(f"  Initial shock position: x₀ = {x0_shock}")
print(f"  Shock position at t={t_final:.4f}: x_shock ≈ {shock_pos_final:.4f}")
print(f"  Distance traveled: {shock_speed * t_final:.4f}")

## Compare Large-Amplitude Numerical and Analytical Solutions

In [ ]:
# Get final density profile from simulation
rho_numerical = binning_data[-1, :]

# Construct analytical shock solution at final time
eta1_fine = np.linspace(0, 1, 1000)
shock_pos = x0_shock + shock_speed * t_final
rho_analytical = np.where(eta1_fine < shock_pos, rho_L, rho_R)

# Plot comparison
fig, ax = plt.subplots(figsize=(12, 6))

# Numerical solution (binned)
ax.step(eta1_bins, rho_numerical, where='mid', label='Numerical (SPH)', linewidth=2, color='blue')

# Analytical weak solution
ax.plot(eta1_fine, rho_analytical, '--', label='Analytical (Rankine-Hugoniot)', linewidth=2, color='red')

# Annotations
ax.axvline(shock_pos_final, color='orange', linestyle=':', linewidth=2, alpha=0.7, label=f'Shock front at t={t_final:.4f}')
ax.set_xlabel(r'$\eta_1$ (logical coordinate)', fontsize=12)
ax.set_ylabel(r'$\rho$ (density)', fontsize=12)
ax.set_title(f'Large-Amplitude Shock: Numerical vs Analytical at t={t_final:.4f}', fontsize=13)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11)
ax.set_ylim([0, 1.2])

plt.tight_layout()
plt.show()

print(f"\nComparison at t={t_final:.4f}:")
print(f"  Numerical shock position (steepest gradient): ≈ {eta1_bins[np.argmin(np.diff(rho_numerical))]}")
print(f"  Analytical shock position: {shock_pos_final:.4f}")

## Analytical Solution: Small-Amplitude Linear Waves

For small-amplitude perturbations, the velocity evolves along characteristics via the method of characteristics:

$$
u(x, t) = \epsilon \sin(\pi(x - \epsilon \pi t)) + O(\epsilon^2)
$$

This comes from solving the characteristic equations:

$$
\frac{dx}{dt} = u, \quad x(0) = x_0
$$

to first order in $\epsilon$.

In [ ]:
# Analytical solution for small-amplitude waves (method of characteristics)
# u(x, t) = epsilon * sin(pi * (x - epsilon * pi * t))

t_final2 = t_grid2[-1]
wave_phase_shift = epsilon_amplitude * np.pi * t_final2

eta1_fine_linear = np.linspace(0, 1, 1000)
u_analytical_linear = epsilon_amplitude * np.sin(np.pi * (eta1_fine_linear - wave_phase_shift))

print("\nAnalytical Small-Amplitude Solution:")
print(f"  Perturbation amplitude: ε = {epsilon_amplitude}")
print(f"  Wave phase shift at t={t_final2:.4f}: δφ = ε π t = {wave_phase_shift:.4f}")
print("  Expected velocity profile: u(η₁,t) = ε sin(π(η₁ - εt))")

# Discussion: Density Evolution in Two Regimes

## Large-Amplitude Shock Regime

In the shock case, density **changes significantly** (from 1.0 to 0.125). This is unavoidable because:

- The continuity equation $\partial_t \rho + \partial_x(\rho u) = 0$ couples density to velocity gradients
- Across the shock, both $\rho$ and $u$ jump, satisfying the Rankine-Hugoniot relation
- The SPH method captures this weak solution via particle redistribution and remapping

The shock propagation speed matches the analytical prediction within SPH resolution limits.

## Small-Amplitude Linear Regime

In the linear wave case with small $\epsilon$, density evolution is **minimized**:

- The density perturbation scales as $O(\epsilon)$ relative to the background density
- For $\epsilon = 0.1$, we expect density variations $<$ 1% (as observed above)
- Velocity evolves along characteristics with only weak nonlinear corrections of $O(\epsilon^2)$

This demonstrates that **amplitude tuning is key**: small perturbations suppress density changes, while shock tests require accepting large density variations as physical.

## SPH Shock Capturing

The particle method naturally captures shocks through:

1. **Particle compression**: Particles pile up across the shock, increasing local density
2. **SPH kernel smoothing**: Smooth kernel averages particle distributions into density profiles
3. **Consistent momentum update**: Velocity is updated from forces implicit in the binned density field
4. **No artificial viscosity required** in this formulation (pressureless fluids have no pressure gradient to stabilize)

## Extension

To further explore:

- Increase `Np_particles` for higher resolution and smoother profiles
- Vary `epsilon_amplitude` in the linear case to observe $O(\epsilon)$ scaling
- Tune kernel parameters in `WeightsParameters()` for different smoothing
- Compare multiple time-stepping schemes via `ButcherTableau` options